# 01 — OULAD DuckDB Ingestion, 9 SQL Data Quality Checks & Returning/New Overlap (`P1-B1`)

This notebook verifies the **real OULAD dataset** (UCI ID 349, CC BY 4.0):
1. Loads all 7 raw CSVs into `data/oulad.duckdb` using `read_csv_auto(..., nullstr='?')`.
2. Runs all 9 SQL Data Quality Checks (8 mandatory + check #9 `unregistration_before_registration`) and logs results transparently.
3. Profiles `imd_band` missing rate and `date_registration` (`NULL` and `>= 14/21/28`).
4. Computes `Returning vs New Students` on `2014B` and `2014J` against Train (`2013B + 2013J`).
5. Verifies consistency against `reports/data_quality.json`.

In [1]:
import json

import pandas as pd
from IPython.display import display

from oulad.config import DB_PATH, RAW_DIR, REPORTS_DIR
from oulad.db import (
    compute_student_overlap,
    get_connection,
    get_registration_and_missing_stats,
    load_raw_tables,
    run_quality_checks,
)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
con = get_connection(DB_PATH)
row_counts = load_raw_tables(RAW_DIR, con)
assert row_counts["studentInfo"] == 32593
assert row_counts["studentVle"] == 10655280
display(pd.Series(row_counts, name="row_count").to_frame())

,row_count
courses,22
assessments,206
vle,6364
studentInfo,32593
studentRegistration,32593
studentAssessment,173912
studentVle,10655280


In [2]:
quality_checks = run_quality_checks(con)
reg_stats = get_registration_and_missing_stats(con)
overlap = compute_student_overlap(con)
con.close()

print("=== 9 SQL Quality Checks (Real OULAD DB) ===")
display(pd.Series(quality_checks, name="n_violations").to_frame())

print("\n=== Registration & Missing Stats ===")
display(pd.Series(reg_stats, name="value").to_frame())

print("\n=== Returning vs New Students (vs Train 2013B + 2013J) ===")
display(pd.DataFrame(overlap).T)

=== 9 SQL Quality Checks (Real OULAD DB) ===


,n_violations
duplicate_student_info_keys,0
invalid_final_result,0
orphan_student_vle_sites,0
orphan_student_assessments,0
invalid_assessment_scores,0
non_positive_sum_click,0
vle_date_after_course_end,0
imd_band_missing_ge_5pct,0
unregistration_before_registration,0



=== Registration & Missing Stats ===


,value
student_info_total_rows,32593.0000
imd_band_missing_count,1111.0000
imd_band_missing_pct,3.4087
date_registration_null_count,45.0000
date_registration_ge_14,59.0000
date_registration_ge_21,26.0000
date_registration_ge_28,18.0000



=== Returning vs New Students (vs Train 2013B + 2013J) ===


,unique_students_total,unique_students_returning,unique_students_new,unique_returning_pct,enrollments_total,enrollments_returning,enrollments_new,enrollments_returning_pct
2014B,7333.0,940.0,6393.0,12.82,7804.0,963.0,6841.0,12.34
2014J,10670.0,972.0,9698.0,9.11,11260.0,999.0,10261.0,8.87


In [3]:
committed_path = REPORTS_DIR / "data_quality.json"
committed = json.loads(committed_path.read_text(encoding="utf-8"))
assert committed["row_counts"] == row_counts
assert committed["quality_checks"] == quality_checks
assert committed["registration_and_missing_stats"] == reg_stats
assert committed["returning_vs_new_students"] == overlap
print(
    f"Verified 100% exact match with committed {committed_path.name} "
    f"(0 violations across {len(quality_checks)} checks)."
)

Verified 100% exact match with committed data_quality.json (0 violations across 9 checks).
